In [ ]:
def analyze_pc_pi_scm_runtimes(
    pc_dir="runtime_PC",
    pi_dir="runtime_RaspberryPi",
    output_dir="runtime_comparison_PC_vs_RaspberryPi",
    search_from=".",
    display_table=True,
    show_plot=True,
    round_digits=3,
):
    """
    Analyze partial or complete SCM runtime results from PC and Raspberry Pi runs.

    This function loads all existing result files generated by your experiment code.
    It does not require both PC and Raspberry Pi results to exist.

    It searches for:
        grid_all_selected_metrics.csv
        all_selected_metrics.csv
        selected_metrics.csv

    Main output:
        dissertation_runtime_summary_df
            Compact dataframe for dissertation use.

        paired_runtime_details_df
            One row per experiment configuration with PC and Pi columns.

        raw_model_runtime_df
            Long-format dataframe with all loaded runtime rows.

        fig
            One plot showing runtime versus control group size.
    """

    from pathlib import Path
    import re
    import warnings

    import matplotlib.pyplot as plt
    import numpy as np
    import pandas as pd

    try:
        from IPython.display import display
    except Exception:
        display = None

    hardware_dirs = {
        "PC": pc_dir,
        "RaspberryPi": pi_dir,
    }

    def _resolve_folder(folder):
        folder = Path(folder)

        if folder.exists():
            return folder

        root = Path(search_from)
        matches = [p for p in root.rglob("*") if p.is_dir() and p.name == folder.name]

        if len(matches) == 0:
            matches = [
                p for p in root.rglob("*")
                if p.is_dir() and p.name.lower() == folder.name.lower()
            ]

        if len(matches) == 0:
            warnings.warn(f"Folder not found: {folder}")
            return folder

        return matches[0]

    def _safe_read_csv(path):
        path = Path(path)

        if not path.exists() or path.stat().st_size == 0:
            return pd.DataFrame()

        try:
            return pd.read_csv(path)
        except Exception as exc:
            warnings.warn(f"Could not read {path}. Reason: {exc}")
            return pd.DataFrame()

    def _parse_metadata_from_path(path):
        path = Path(path)

        pattern = re.compile(
            r"features=(?P<feature_mode>.*?)__controls=(?P<controls>.*?)__rep=(?P<rep>\d+)"
        )

        metadata = {
            "experiment_id": np.nan,
            "feature_mode": np.nan,
            "control_group_size_label": np.nan,
            "repetition": np.nan,
            "dataset_from_path": np.nan,
        }

        for part in reversed(path.parts):
            match = pattern.fullmatch(part)

            if match is not None:
                metadata["experiment_id"] = part
                metadata["feature_mode"] = match.group("feature_mode")
                metadata["control_group_size_label"] = match.group("controls")
                metadata["repetition"] = int(match.group("rep"))
                break

        if path.name == "selected_metrics.csv":
            metadata["dataset_from_path"] = path.parent.name

        return metadata

    def _add_metadata(df, metadata):
        df = df.copy()

        if "dataset" not in df.columns:
            df["dataset"] = metadata["dataset_from_path"]
        else:
            df["dataset"] = df["dataset"].fillna(metadata["dataset_from_path"])

        for column in [
            "experiment_id",
            "feature_mode",
            "control_group_size_label",
            "repetition",
        ]:
            if column not in df.columns:
                df[column] = metadata[column]
            else:
                df[column] = df[column].fillna(metadata[column])

        return df

    def _normalise_control_label(value):
        if pd.isna(value):
            return "unknown"

        value = str(value)

        if value.lower() == "all":
            return "all"

        try:
            return str(int(float(value)))
        except Exception:
            return value

    def _coerce_numeric(df, columns):
        df = df.copy()

        for column in columns:
            if column not in df.columns:
                df[column] = np.nan

            df[column] = pd.to_numeric(df[column], errors="coerce")

        return df

    def _add_runtime_columns(df):
        runtime_columns = [
            "selection_total_seconds",
            "selection_fit_seconds",
            "selection_total_inference_seconds",
            "selected_model_total_seconds",
            "test_stage_total_seconds",
            "pre_post_stage_total_seconds",
            "test_lag_construction_seconds",
            "test_design_seconds",
            "test_refit_seconds",
            "test_inference_seconds",
            "pre_lag_construction_seconds",
            "pre_design_seconds",
            "pre_refit_seconds",
            "pre_inference_seconds",
            "post_inference_seconds",
            "n_used_current_controls",
            "n_features",
        ]

        df = _coerce_numeric(df, runtime_columns)

        missing_selected_total = df["selected_model_total_seconds"].isna()
        derived_selected_total = (
            df["test_stage_total_seconds"]
            + df["pre_post_stage_total_seconds"]
        )

        df.loc[missing_selected_total, "selected_model_total_seconds"] = derived_selected_total.loc[
            missing_selected_total
        ]

        df["total_model_runtime_seconds"] = (
            df["selection_total_seconds"].fillna(0.0)
            + df["selected_model_total_seconds"].fillna(0.0)
        )

        no_runtime = (
            df["selection_total_seconds"].isna()
            & df["selected_model_total_seconds"].isna()
            & df["test_stage_total_seconds"].isna()
            & df["pre_post_stage_total_seconds"].isna()
        )

        df.loc[no_runtime, "total_model_runtime_seconds"] = np.nan

        return df

    def _load_hardware_folder(hardware, folder):
        folder = _resolve_folder(folder)

        if not folder.exists():
            return pd.DataFrame()

        candidate_files = []

        grid_file = folder / "grid_all_selected_metrics.csv"
        if grid_file.exists():
            candidate_files.append((grid_file, 0, "grid_all_selected_metrics"))

        for path in sorted(folder.rglob("all_selected_metrics.csv")):
            candidate_files.append((path, 1, "all_selected_metrics"))

        for path in sorted(folder.rglob("selected_metrics.csv")):
            candidate_files.append((path, 2, "selected_metrics"))

        frames = []

        for path, source_priority, source_kind in candidate_files:
            df = _safe_read_csv(path)

            if df.empty:
                continue

            metadata = _parse_metadata_from_path(path)
            df = _add_metadata(df, metadata)

            df["hardware"] = hardware
            df["source_path"] = str(path)
            df["source_kind"] = source_kind
            df["source_priority"] = source_priority

            frames.append(df)

        if len(frames) == 0:
            return pd.DataFrame()

        out = pd.concat(frames, ignore_index=True)

        for column, default in [
            ("dataset", "unknown"),
            ("model", "unknown"),
            ("feature_mode", "unknown"),
            ("control_group_size_label", "unknown"),
            ("repetition", 0),
        ]:
            if column not in out.columns:
                out[column] = default

        out["dataset"] = out["dataset"].fillna("unknown").astype(str)
        out["model"] = out["model"].fillna("unknown").astype(str)
        out["feature_mode"] = out["feature_mode"].fillna("unknown").astype(str)
        out["control_group_size_label"] = out["control_group_size_label"].apply(
            _normalise_control_label
        )
        out["repetition"] = pd.to_numeric(
            out["repetition"], errors="coerce"
        ).fillna(0).astype(int)

        out = _add_runtime_columns(out)

        dedup_columns = [
            "hardware",
            "dataset",
            "model",
            "feature_mode",
            "control_group_size_label",
            "repetition",
        ]

        out = (
            out.sort_values("source_priority")
            .drop_duplicates(subset=dedup_columns, keep="first")
            .reset_index(drop=True)
        )

        return out

    raw_frames = []

    for hardware, folder in hardware_dirs.items():
        df = _load_hardware_folder(hardware, folder)

        if not df.empty:
            raw_frames.append(df)

    if len(raw_frames) == 0:
        return pd.DataFrame(), pd.DataFrame(), pd.DataFrame(), None

    raw_model_runtime_df = pd.concat(raw_frames, ignore_index=True)

    key_columns = [
        "dataset",
        "model",
        "feature_mode",
        "control_group_size_label",
        "repetition",
    ]

    descriptive_columns = [
        "kind",
        "treated_unit",
        "n_used_current_controls",
        "n_features",
        "n_effective_pre",
        "n_training",
        "n_validation",
        "n_test",
        "n_post",
        "lambda",
        "sum_to_one_constraint",
        "positive_weights_constraint",
        "best_epoch",
        "standardize_features",
    ]

    for column in descriptive_columns:
        if column not in raw_model_runtime_df.columns:
            raw_model_runtime_df[column] = np.nan

    base_df = (
        raw_model_runtime_df[key_columns + descriptive_columns]
        .drop_duplicates(subset=key_columns)
        .reset_index(drop=True)
    )

    paired_runtime_details_df = base_df.copy()

    runtime_columns = [
        "selection_total_seconds",
        "selected_model_total_seconds",
        "total_model_runtime_seconds",
        "test_stage_total_seconds",
        "pre_post_stage_total_seconds",
        "pre_refit_seconds",
        "post_inference_seconds",
        "source_path",
        "source_kind",
    ]

    for hardware in ["PC", "RaspberryPi"]:
        hardware_df = raw_model_runtime_df[
            raw_model_runtime_df["hardware"] == hardware
        ].copy()

        for column in runtime_columns:
            if column not in hardware_df.columns:
                hardware_df[column] = np.nan

        hardware_df = hardware_df[key_columns + runtime_columns].drop_duplicates(
            subset=key_columns
        )

        hardware_df = hardware_df.rename(
            columns={
                column: f"{hardware}_{column}"
                for column in runtime_columns
            }
        )

        paired_runtime_details_df = paired_runtime_details_df.merge(
            hardware_df,
            on=key_columns,
            how="outer",
        )

    if "PC_total_model_runtime_seconds" not in paired_runtime_details_df.columns:
        paired_runtime_details_df["PC_total_model_runtime_seconds"] = np.nan

    if "RaspberryPi_total_model_runtime_seconds" not in paired_runtime_details_df.columns:
        paired_runtime_details_df["RaspberryPi_total_model_runtime_seconds"] = np.nan

    paired_runtime_details_df["runtime_ratio_pi_over_pc"] = (
        paired_runtime_details_df["RaspberryPi_total_model_runtime_seconds"]
        / paired_runtime_details_df["PC_total_model_runtime_seconds"]
    )

    paired_runtime_details_df["has_pc_result"] = paired_runtime_details_df[
        "PC_total_model_runtime_seconds"
    ].notna()

    paired_runtime_details_df["has_pi_result"] = paired_runtime_details_df[
        "RaspberryPi_total_model_runtime_seconds"
    ].notna()

    paired_runtime_details_df["has_matched_result"] = (
        paired_runtime_details_df["has_pc_result"]
        & paired_runtime_details_df["has_pi_result"]
    )

    paired_runtime_details_df["comparison_status"] = np.select(
        [
            paired_runtime_details_df["has_matched_result"],
            paired_runtime_details_df["has_pc_result"]
            & ~paired_runtime_details_df["has_pi_result"],
            ~paired_runtime_details_df["has_pc_result"]
            & paired_runtime_details_df["has_pi_result"],
        ],
        [
            "matched",
            "only_pc_finished",
            "only_pi_finished",
        ],
        default="missing",
    )

    def _model_sort(value):
        return {"SCM": 0, "NN-SCM": 1}.get(str(value), 99)

    def _feature_sort(value):
        return {
            "standard_donors": 0,
            "treated_lags": 1,
            "treated_and_donor_lags": 2,
        }.get(str(value), 99)

    def _control_sort(value):
        value = _normalise_control_label(value)

        if value == "all":
            return float("inf")

        try:
            return float(value)
        except Exception:
            return float("inf") - 1.0

    paired_runtime_details_df["_model_sort"] = paired_runtime_details_df["model"].apply(
        _model_sort
    )
    paired_runtime_details_df["_feature_sort"] = paired_runtime_details_df[
        "feature_mode"
    ].apply(_feature_sort)
    paired_runtime_details_df["_control_sort"] = paired_runtime_details_df[
        "control_group_size_label"
    ].apply(_control_sort)

    paired_runtime_details_df = (
        paired_runtime_details_df
        .sort_values(
            [
                "dataset",
                "_model_sort",
                "_feature_sort",
                "_control_sort",
                "repetition",
            ]
        )
        .drop(columns=["_model_sort", "_feature_sort", "_control_sort"])
        .reset_index(drop=True)
    )

    summary_group_columns = [
        "dataset",
        "model",
        "feature_mode",
        "control_group_size_label",
    ]

    dissertation_runtime_summary_df = (
        paired_runtime_details_df
        .groupby(summary_group_columns, dropna=False)
        .agg(
            n_repetitions=("repetition", "nunique"),
            n_pc_results=("has_pc_result", "sum"),
            n_pi_results=("has_pi_result", "sum"),
            n_matched_results=("has_matched_result", "sum"),
            n_used_current_controls=("n_used_current_controls", "first"),
            n_features_mean=("n_features", "mean"),
            pc_runtime_seconds_mean=("PC_total_model_runtime_seconds", "mean"),
            pc_runtime_seconds_std=("PC_total_model_runtime_seconds", "std"),
            pi_runtime_seconds_mean=("RaspberryPi_total_model_runtime_seconds", "mean"),
            pi_runtime_seconds_std=("RaspberryPi_total_model_runtime_seconds", "std"),
            pi_over_pc_runtime_ratio_mean=("runtime_ratio_pi_over_pc", "mean"),
        )
        .reset_index()
    )

    dissertation_runtime_summary_df["_model_sort"] = dissertation_runtime_summary_df[
        "model"
    ].apply(_model_sort)
    dissertation_runtime_summary_df["_feature_sort"] = dissertation_runtime_summary_df[
        "feature_mode"
    ].apply(_feature_sort)
    dissertation_runtime_summary_df["_control_sort"] = dissertation_runtime_summary_df[
        "control_group_size_label"
    ].apply(_control_sort)

    dissertation_runtime_summary_df = (
        dissertation_runtime_summary_df
        .sort_values(
            [
                "dataset",
                "_model_sort",
                "_feature_sort",
                "_control_sort",
            ]
        )
        .drop(columns=["_model_sort", "_feature_sort", "_control_sort"])
        .reset_index(drop=True)
    )

    numeric_summary_columns = dissertation_runtime_summary_df.select_dtypes(
        include=[np.number]
    ).columns

    dissertation_runtime_summary_df[numeric_summary_columns] = (
        dissertation_runtime_summary_df[numeric_summary_columns]
        .round(round_digits)
    )

    numeric_detail_columns = paired_runtime_details_df.select_dtypes(
        include=[np.number]
    ).columns

    paired_runtime_details_df[numeric_detail_columns] = (
        paired_runtime_details_df[numeric_detail_columns]
        .round(round_digits)
    )

    if display_table and display is not None:
        display(dissertation_runtime_summary_df)

    plot_df = (
        raw_model_runtime_df
        .dropna(subset=["total_model_runtime_seconds"])
        .copy()
    )

    fig = None

    if not plot_df.empty:
        plot_df["control_group_size_numeric"] = pd.to_numeric(
            plot_df["n_used_current_controls"],
            errors="coerce",
        )

        missing_x = plot_df["control_group_size_numeric"].isna()

        plot_df.loc[missing_x, "control_group_size_numeric"] = (
            plot_df.loc[missing_x, "control_group_size_label"]
            .replace({"all": np.nan, "unknown": np.nan})
            .pipe(pd.to_numeric, errors="coerce")
        )

        plot_df = plot_df.dropna(subset=["control_group_size_numeric"])

        aggregated_plot_df = (
            plot_df
            .groupby(
                [
                    "hardware",
                    "dataset",
                    "model",
                    "feature_mode",
                    "control_group_size_label",
                    "control_group_size_numeric",
                ],
                dropna=False,
            )
            .agg(
                runtime_seconds_mean=("total_model_runtime_seconds", "mean"),
                runtime_seconds_std=("total_model_runtime_seconds", "std"),
                n_results=("total_model_runtime_seconds", "count"),
            )
            .reset_index()
        )

        if not aggregated_plot_df.empty:
            fig, ax = plt.subplots(figsize=(11.5, 6.0))

            line_groups = [
                "hardware",
                "model",
                "feature_mode",
            ]

            for group_values, group_df in aggregated_plot_df.groupby(line_groups):
                hardware, model, feature_mode = group_values

                group_df = group_df.sort_values("control_group_size_numeric")

                label = f"{hardware} | {model} | {feature_mode}"

                ax.plot(
                    group_df["control_group_size_numeric"],
                    group_df["runtime_seconds_mean"],
                    marker="o",
                    label=label,
                )

            ax.set_xlabel("Number of control units")
            ax.set_ylabel("Mean runtime in seconds")
            ax.set_title("SCM runtime versus control group size")
            ax.grid(True, alpha=0.25)
            ax.legend(fontsize=8, frameon=True)

            fig.tight_layout()

            if show_plot:
                plt.show()

    if output_dir is not None:
        output_dir = Path(output_dir)
        output_dir.mkdir(parents=True, exist_ok=True)

        dissertation_runtime_summary_df.to_csv(
            output_dir / "pc_pi_runtime_summary_for_dissertation.csv",
            index=False,
        )

        paired_runtime_details_df.to_csv(
            output_dir / "pc_pi_runtime_paired_details.csv",
            index=False,
        )

        raw_model_runtime_df.to_csv(
            output_dir / "pc_pi_runtime_raw_loaded_rows.csv",
            index=False,
        )

        if fig is not None:
            fig.savefig(
                output_dir / "runtime_vs_control_group_size.png",
                dpi=300,
                bbox_inches="tight",
            )

    return (
        dissertation_runtime_summary_df,
        paired_runtime_details_df,
        raw_model_runtime_df,
        fig,
    )

(
    dissertation_runtime_summary_df,
    paired_runtime_details_df,
    raw_model_runtime_df,
    fig,
) = analyze_pc_pi_scm_runtimes(
    pc_dir="runtime_PC",
    pi_dir="runtime_RaspberryPi",
    output_dir="runtime_comparison_PC_vs_RaspberryPi",
    search_from=".",
    display_table=True,
    show_plot=True,
)